# LeetCode 976 — Largest Perimeter Triangle bằng Python

**Link đề:** https://leetcode.com/problems/largest-perimeter-triangle/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Greedy, Sorting

```python
class Solution:
    def largestPerimeter(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Nắm **bất đẳng thức tam giác** và mẹo rút gọn: với `a ≤ b ≤ c`, chỉ cần kiểm tra **một** điều kiện `a + b > c`.
2. Ôn `sort` / `sorted` (tăng, giảm với `reverse=True`), `itertools.combinations`, `heapq`.
3. Hiểu lập luận **tham lam** (greedy): sau khi sắp xếp giảm dần, chỉ cần xét **bộ ba liên tiếp**.
4. Viết 4 cách: brute force mọi bộ ba, sắp xếp + bộ ba liên tiếp, duyệt tăng dần từ cuối, heap lấy dần phần tử lớn nhất.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Cho mảng số nguyên dương `nums` (độ dài các đoạn thẳng). Chọn **3** đoạn tạo thành tam giác có **diện tích khác 0** sao cho **chu vi lớn nhất**; trả về chu vi đó. Nếu không thể tạo tam giác nào, trả về `0`.

| nums | Kết quả | Giải thích |
|---|:---:|---|
| `[2,1,2]` | `5` | 1, 2, 2 tạo được tam giác: 1+2+2 = 5 |
| `[1,2,1,10]` | `0` | (1,1,2), (1,1,10), (1,2,10) đều **không** tạo được tam giác |

**Constraints:** `3 <= len(nums) <= 10^4`, `1 <= nums[i] <= 10^6`.

"Diện tích khác 0" nghĩa là tam giác **không suy biến**: ba đoạn `1, 1, 2` nằm thẳng hàng (1 + 1 = 2) **không** được tính.

## 2. Nền tảng toán và Python

### 2.1. Bất đẳng thức tam giác

Ba độ dài `a, b, c > 0` tạo tam giác (diện tích > 0) ⇔ **cả ba** điều kiện:

```text
a + b > c,   a + c > b,   b + c > a
```

**Mẹo rút gọn:** nếu đã sắp `a ≤ b ≤ c` thì `a + c > b` và `b + c > a` **luôn đúng** (vì `c ≥ b` và `a > 0` → `a + c > b`; tương tự). Chỉ còn **một** điều kiện cần kiểm:

```text
a + b > c     (tổng hai cạnh nhỏ > cạnh lớn nhất)
```

In [1]:
def is_triangle(a: int, b: int, c: int) -> bool:
    a, b, c = sorted((a, b, c))     # sắp để c là cạnh lớn nhất
    return a + b > c

for sides in [(1, 2, 2), (1, 1, 2), (3, 4, 5), (1, 2, 10), (5, 5, 5)]:
    print(sides, "->", is_triangle(*sides))

(1, 2, 2) -> True
(1, 1, 2) -> False
(3, 4, 5) -> True
(1, 2, 10) -> False
(5, 5, 5) -> True


`is_triangle(*sides)` dùng **toán tử giải nén** `*`: tuple `(1, 2, 2)` được "bung" thành 3 đối số `a=1, b=2, c=2`.

Dòng `a, b, c = sorted((a, b, c))` là **gán nhiều biến cùng lúc** (unpacking): list 3 phần tử được tách vào 3 biến.

### 2.2. Sắp xếp trong Python

- `sorted(nums)` trả về **list mới** đã sắp tăng dần, không sửa `nums`.
- `nums.sort()` sắp **tại chỗ** (in-place), trả về `None`.
- Thêm `reverse=True` để sắp giảm dần.

In [2]:
nums = [3, 6, 2, 3]
print("sorted:", sorted(nums), "| nums vẫn là", nums)
print("giảm dần:", sorted(nums, reverse=True))
nums.sort()
print("sau nums.sort():", nums)
print("nums.sort() trả về:", [3, 1].sort())

sorted: [2, 3, 3, 6] | nums vẫn là [3, 6, 2, 3]
giảm dần: [6, 3, 3, 2]
sau nums.sort(): [2, 3, 3, 6]
nums.sort() trả về: None


### 2.3. `itertools.combinations` — sinh mọi bộ chọn

`combinations(arr, 3)` sinh mọi bộ 3 phần tử ở **3 vị trí khác nhau** (không xét thứ tự). Có `C(n, 3) = n(n−1)(n−2)/6` bộ.

In [3]:
from itertools import combinations
print(list(combinations([2, 1, 2], 3)))
print(list(combinations([1, 2, 1, 10], 3)))
print("Số bộ ba với n=100:", len(list(combinations(range(100), 3))))

[(2, 1, 2)]
[(1, 2, 1), (1, 2, 10), (1, 1, 10), (2, 1, 10)]
Số bộ ba với n=100: 161700


## 3. Cách trực giác (baseline) — thử mọi bộ ba

In [4]:
def largest_perimeter_brute(nums: list[int]) -> int:
    best = 0
    for a, b, c in combinations(nums, 3):
        if is_triangle(a, b, c):
            best = max(best, a + b + c)
    return best

print(largest_perimeter_brute([2, 1, 2]))
print(largest_perimeter_brute([1, 2, 1, 10]))
print(largest_perimeter_brute([3, 6, 2, 3]))

5
0
8


Độ phức tạp: **O(n³)** bộ ba. Với `n = 10^4` là khoảng 1.7·10¹¹ bộ — không thể chạy kịp.

## 4. Nút thắt

Ta đang thử những bộ ba **chắc chắn không thể là đáp án**. Ví dụ, nếu cạnh lớn nhất là `c`, thì muốn chu vi lớn **và** muốn `a + b > c` dễ thoả, ta nên chọn `a, b` **càng lớn càng tốt** (nhưng ≤ `c`). Chọn hai cạnh nhỏ hơn chỉ làm tổng `a + b` nhỏ đi → vừa khó thoả điều kiện, vừa giảm chu vi.

## 5. Insight — chỉ cần xét bộ ba liên tiếp sau khi sắp xếp

Sắp `nums` **giảm dần**: `s[0] ≥ s[1] ≥ s[2] ≥ ...`

Xét cạnh lớn nhất là `s[i]`. Hai cạnh còn lại tốt nhất là **hai số lớn nhất đứng sau nó**: `s[i+1]` và `s[i+2]`.
- Nếu `s[i+1] + s[i+2] > s[i]` → bộ `(s[i], s[i+1], s[i+2])` là tam giác có chu vi lớn nhất **với cạnh lớn nhất `s[i]`**.
- Nếu **không** thoả → **không** có cặp nào khác sau `s[i]` thoả (mọi cặp khác có tổng ≤ `s[i+1] + s[i+2]`) → `s[i]` không thể làm cạnh lớn nhất, bỏ nó.

Duyệt `i = 0, 1, 2, ...`: bộ ba **đầu tiên** thoả chính là đáp án, vì chu vi các bộ sau không lớn hơn (từng cạnh đều ≤).

In [5]:
s = sorted([3, 6, 2, 3], reverse=True)
print("giảm dần:", s)
for i in range(len(s) - 2):
    print(f"i={i}: bộ ({s[i]}, {s[i+1]}, {s[i+2]}) -> {s[i+1]} + {s[i+2]} = {s[i+1] + s[i+2]} > {s[i]}? {s[i+1] + s[i+2] > s[i]}")

giảm dần: [6, 3, 3, 2]
i=0: bộ (6, 3, 3) -> 3 + 3 = 6 > 6? False
i=1: bộ (3, 3, 2) -> 3 + 2 = 5 > 3? True


## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1:** sắp giảm dần: `nums.sort(reverse=True)`.

**Đoạn 2:** duyệt `i` từ `0` đến `len(nums) - 3` (để `i + 2` còn trong mảng): `range(len(nums) - 2)`.

**Đoạn 3:** nếu `nums[i+1] + nums[i+2] > nums[i]` → trả chu vi ngay.

**Đoạn 4:** hết vòng mà không thấy → `return 0`.

In [6]:
def largest_perimeter(nums: list[int]) -> int:
    nums = sorted(nums, reverse=True)            # Đoạn 1 (bản sao, không sửa input)
    for i in range(len(nums) - 2):               # Đoạn 2
        if nums[i + 1] + nums[i + 2] > nums[i]:  # Đoạn 3
            return nums[i] + nums[i + 1] + nums[i + 2]
    return 0                                     # Đoạn 4

print(largest_perimeter([2, 1, 2]))
print(largest_perimeter([1, 2, 1, 10]))
print(largest_perimeter([3, 6, 2, 3]))

5
0
8


## 7. Trace từng bước

Ví dụ `nums = [1, 2, 1, 10]` → giảm dần `[10, 2, 1, 1]`:

| i | c = s[i] | b = s[i+1] | a = s[i+2] | a + b | a + b > c? | Hành động |
|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 0 | 10 | 2 | 1 | 3 | không | bỏ 10 |
| 1 | 2 | 1 | 1 | 2 | không (bằng!) | bỏ 2 |
| — | | | | | | hết vòng → **0** |

Ví dụ `[3, 6, 2, 3]` → `[6, 3, 3, 2]`: `i=0`: 3+3=6 không > 6 → bỏ; `i=1`: 3+2=5 > 3 → trả **8**.

In [7]:
def trace_largest(nums: list[int]) -> int:
    s = sorted(nums, reverse=True)
    print("Sắp giảm dần:", s)
    for i in range(len(s) - 2):
        c, b_, a = s[i], s[i + 1], s[i + 2]
        ok = a + b_ > c
        print(f"  i={i}: c={c}, b={b_}, a={a}, a+b={a + b_} {'>' if ok else '<='} {c}", end=" ")
        if ok:
            print(f"-> tam giác! chu vi = {a + b_ + c}")
            return a + b_ + c
        print("-> bỏ cạnh", c)
    print("  Không có bộ nào -> 0")
    return 0

trace_largest([1, 2, 1, 10])
print()
trace_largest([3, 6, 2, 3])
print()
trace_largest([100, 1, 50, 49, 30, 25])

Sắp giảm dần: [10, 2, 1, 1]
  i=0: c=10, b=2, a=1, a+b=3 <= 10 -> bỏ cạnh 10
  i=1: c=2, b=1, a=1, a+b=2 <= 2 -> bỏ cạnh 2
  Không có bộ nào -> 0

Sắp giảm dần: [6, 3, 3, 2]
  i=0: c=6, b=3, a=3, a+b=6 <= 6 -> bỏ cạnh 6
  i=1: c=3, b=3, a=2, a+b=5 > 3 -> tam giác! chu vi = 8

Sắp giảm dần: [100, 50, 49, 30, 25, 1]
  i=0: c=100, b=50, a=49, a+b=99 <= 100 -> bỏ cạnh 100
  i=1: c=50, b=49, a=30, a+b=79 > 50 -> tam giác! chu vi = 129


129

Ví dụ thứ ba: cạnh 100 bị bỏ (50+49 = 99 ≤ 100), sau đó `(50, 49, 30)` thoả → chu vi 129.

## 8. Code nộp LeetCode

### Cách A — sắp giảm dần + bộ ba liên tiếp

In [8]:
class Solution:
    def largestPerimeter(self, nums: list[int]) -> int:
        nums.sort(reverse=True)
        for i in range(len(nums) - 2):
            if nums[i + 1] + nums[i + 2] > nums[i]:
                return nums[i] + nums[i + 1] + nums[i + 2]
        return 0

print(Solution().largestPerimeter([2, 1, 2]))
print(Solution().largestPerimeter([1, 2, 1, 10]))

5
0


Lưu ý: `nums.sort()` sửa trực tiếp mảng đầu vào. Trên LeetCode điều này chấp nhận được; trong code thực tế nên dùng `sorted` nếu người gọi còn cần mảng gốc.

### Cách B — sắp tăng dần, duyệt từ cuối về

In [9]:
class SolutionAsc:
    def largestPerimeter(self, nums: list[int]) -> int:
        nums = sorted(nums)
        for i in range(len(nums) - 1, 1, -1):       # i = n-1, n-2, ..., 2
            if nums[i - 2] + nums[i - 1] > nums[i]:
                return nums[i - 2] + nums[i - 1] + nums[i]
        return 0

print(list(range(5, 1, -1)))
print(SolutionAsc().largestPerimeter([3, 6, 2, 3]))

[5, 4, 3, 2]
8


`range(start, stop, step)` với `step = -1` đếm **lùi**: `range(5, 1, -1)` → `5, 4, 3, 2` (không gồm `stop = 1`).

### Cách C — heap: chỉ "rút" các phần tử lớn nhất khi cần

Nếu đáp án thường tìm được ở gần đầu dãy giảm dần, sắp xếp **toàn bộ** là thừa. `heapq` là **min-heap**: `heapify` biến list thành heap trong O(n), `heappop` lấy phần tử **nhỏ nhất** trong O(log n). Muốn lấy lớn nhất → lưu số **âm**.

In [10]:
import heapq
h = [-x for x in [3, 6, 2, 3]]
heapq.heapify(h)
print("rút dần số lớn nhất:", [-heapq.heappop(h) for _ in range(4)])

rút dần số lớn nhất: [6, 3, 3, 2]


In [11]:
class SolutionHeap:
    def largestPerimeter(self, nums: list[int]) -> int:
        h = [-x for x in nums]
        heapq.heapify(h)                       # O(n)
        c = -heapq.heappop(h)
        b = -heapq.heappop(h)
        while h:
            a = -heapq.heappop(h)              # mỗi lần O(log n)
            if a + b > c:
                return a + b + c
            c, b = b, a                        # trượt cửa sổ 3 phần tử
        return 0

print(SolutionHeap().largestPerimeter([100, 1, 50, 49, 30, 25]))
print(SolutionHeap().largestPerimeter([1, 2, 1, 10]))

129
0


`c, b = b, a` là **gán đồng thời**: vế phải được tính xong (lấy giá trị cũ của `b`, `a`) rồi mới gán — không cần biến tạm.

## 9. Vì sao đúng?

Gọi `s` là dãy giảm dần. Bổ đề: với mỗi `i`, trong mọi tam giác có cạnh lớn nhất là `s[i]` (chọn từ các phần tử sau `i`), bộ `(s[i], s[i+1], s[i+2])` có chu vi lớn nhất, và nếu bộ đó không phải tam giác thì không có tam giác nào nhận `s[i]` làm cạnh lớn nhất.
- Mọi cặp `(s[j], s[k])` với `i < j < k` có `s[j] ≤ s[i+1]`, `s[k] ≤ s[i+2]` → tổng ≤ `s[i+1] + s[i+2]`.

**Invariant khi duyệt:** trước khi xét `i`, mọi phần tử `s[0..i-1]` đã được chứng minh không thể là cạnh lớn nhất của tam giác nào. Do đó tam giác tốt nhất (nếu có) có cạnh lớn nhất là `s[i]` hoặc nhỏ hơn. Bộ thoả đầu tiên có từng cạnh ≥ cạnh tương ứng của mọi bộ sau → chu vi lớn nhất. Hết vòng → không có tam giác → `0`.

*Tại sao chỉ cần một bất đẳng thức?* Xem mục 2.1: với `a ≤ b ≤ c` và độ dài dương, hai bất đẳng thức còn lại tự đúng.

## 10. Độ phức tạp

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Brute force | O(n³) | O(1) | chỉ dùng kiểm chứng |
| A. Sort giảm + bộ ba | O(n log n) | O(1) thêm (Timsort cần tối đa O(n) tạm) | chuẩn |
| B. Sort tăng, duyệt ngược | O(n log n) | O(n) (`sorted` tạo bản sao) | tương đương A |
| C. Heap | O(n + k log n), `k` = số lần rút | O(n) | nhanh khi đáp án ở gần đầu; xấu nhất O(n log n) |

Trường hợp xấu cho mọi cách: dãy kiểu Fibonacci (1, 1, 2, 3, 5, 8, ...) — mỗi bộ ba liên tiếp có `a + b = c` nên phải duyệt hết.

## 11. Bộ test

In [12]:
fib = [1, 1]
while len(fib) < 25:
    fib.append(fib[-1] + fib[-2])

tests = [
    ([2, 1, 2], 5),
    ([1, 2, 1, 10], 0),
    ([3, 6, 2, 3], 8),
    ([3, 2, 3, 4], 10),
    ([1, 1, 1], 3),                    # tam giác đều nhỏ nhất
    ([1, 1, 2], 0),                    # suy biến (thẳng hàng)
    ([100, 1, 50, 49, 30, 25], 129),
    (fib, 0),                          # Fibonacci: không bộ nào thoả
    ([10**6] * 3, 3 * 10**6),          # giá trị lớn nhất
]
impls = {
    "brute": largest_perimeter_brute, "chính": largest_perimeter,
    "LC-A": lambda a: Solution().largestPerimeter(list(a)),
    "LC-B": SolutionAsc().largestPerimeter, "LC-C": SolutionHeap().largestPerimeter,
}
for nums, expected in tests:
    for name, f in impls.items():
        got = f(nums)
        assert got == expected, (name, nums, got, expected)
    print(f"{str(nums)[:40]:42} -> {expected} PASS")
print("Tất cả test PASS!")

[2, 1, 2]                                  -> 5 PASS
[1, 2, 1, 10]                              -> 0 PASS
[3, 6, 2, 3]                               -> 8 PASS
[3, 2, 3, 4]                               -> 10 PASS
[1, 1, 1]                                  -> 3 PASS
[1, 1, 2]                                  -> 0 PASS
[100, 1, 50, 49, 30, 25]                   -> 129 PASS
[1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 1   -> 0 PASS
[1000000, 1000000, 1000000]                -> 3000000 PASS
Tất cả test PASS!


Chú ý: `Solution().largestPerimeter` sửa mảng tại chỗ, nên ta truyền `list(a)` (bản sao) để không làm hỏng test dùng chung.

In [13]:
import random
rng = random.Random(976)
for _ in range(1500):
    nums = [rng.randint(1, 30) for _ in range(rng.randint(3, 9))]
    expected = largest_perimeter_brute(nums)
    for name, f in impls.items():
        assert f(nums) == expected, (name, nums)
print("1500 mảng ngẫu nhiên: mọi cách khớp brute force")

1500 mảng ngẫu nhiên: mọi cách khớp brute force


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` thay vì `>`

In [14]:
def wrong_ge(nums):
    s = sorted(nums, reverse=True)
    for i in range(len(s) - 2):
        if s[i + 1] + s[i + 2] >= s[i]:      # sai: chấp nhận tam giác suy biến
            return s[i] + s[i + 1] + s[i + 2]
    return 0

print("Sai :", wrong_ge([1, 1, 2]))
print("Đúng:", largest_perimeter([1, 1, 2]))

Sai : 4
Đúng: 0


**Vì sao sai:** `1 + 1 = 2` → ba đoạn thẳng hàng, diện tích bằng 0. **Sửa:** dùng `>`.

### Lỗi 2: Sắp tăng dần nhưng lại duyệt từ đầu và trả bộ đầu tiên

In [15]:
def wrong_ascending_first(nums):
    s = sorted(nums)
    for i in range(len(s) - 2):
        if s[i] + s[i + 1] > s[i + 2]:
            return s[i] + s[i + 1] + s[i + 2]      # tam giác NHỎ nhất, không phải lớn nhất
    return 0

print("Sai :", wrong_ascending_first([2, 3, 4, 5, 6]))
print("Đúng:", largest_perimeter([2, 3, 4, 5, 6]))

Sai : 9
Đúng: 15


**Sửa:** sắp giảm dần (hoặc tăng dần nhưng duyệt từ cuối như Cách B).

### Lỗi 3: Kiểm tra sai cặp cạnh

In [16]:
def wrong_pair(nums):
    s = sorted(nums, reverse=True)
    for i in range(len(s) - 2):
        if s[i] + s[i + 1] > s[i + 2]:            # luôn đúng vì s[i] >= s[i+2]!
            return s[i] + s[i + 1] + s[i + 2]
    return 0

print("Sai :", wrong_pair([1, 2, 1, 10]))
print("Đúng:", largest_perimeter([1, 2, 1, 10]))

Sai : 13
Đúng: 0


**Vì sao sai:** phải lấy **hai cạnh nhỏ** cộng lại so với **cạnh lớn nhất**. Trong dãy giảm dần, cạnh lớn nhất là `s[i]`.

### Lỗi 4: `range(len(nums))` → `IndexError` ở `nums[i+2]`

Vòng phải dừng ở `len(nums) - 2` (không gồm) để `i + 2 ≤ len(nums) − 1`.

### Lỗi 5: Viết `nums = nums.sort()`

`sort()` trả `None`, nên `nums` thành `None` → lỗi ở dòng sau. Dùng `nums.sort()` (không gán) hoặc `nums = sorted(nums)`.

## 13. Mẹo nhận dạng pattern

- "Chọn k phần tử để **tối đa** tổng, kèm một **ràng buộc so sánh** giữa chúng" → **sắp xếp** rồi xét các phần tử **liền kề** (greedy).
- Bất đẳng thức đối xứng (như tam giác) → sắp xếp để chỉ còn **một** điều kiện quyết định.
- Lời giải greedy cần lý lẽ "đổi chỗ" (exchange argument): thay một phần tử bằng phần tử lớn hơn liền kề không làm tệ đi.
- Chỉ cần vài phần tử lớn nhất → cân nhắc `heapq` / `heapq.nlargest` thay vì sắp toàn bộ.

## 14. Biến thể

1. **LeetCode 611 — Valid Triangle Number:** **đếm** số bộ ba tạo tam giác → sắp xếp + hai con trỏ, O(n²).
2. **Tam giác chu vi nhỏ nhất:** sắp tăng dần, bộ ba liên tiếp đầu tiên thoả? *Cẩn thận* — không còn đúng! Với cạnh lớn nhất `c` cố định, muốn chu vi nhỏ phải chọn `a, b` nhỏ nhưng vẫn `a + b > c` — cần hai con trỏ.
3. **LeetCode 2971 — Find Polygon With the Largest Perimeter:** đa giác k cạnh: cạnh lớn nhất < tổng các cạnh còn lại → sắp tăng dần + tổng tiền tố.

In [17]:
def count_triangles(nums):
    s = sorted(nums)
    count = 0
    for k in range(len(s) - 1, 1, -1):        # s[k] là cạnh lớn nhất
        i, j = 0, k - 1
        while i < j:
            if s[i] + s[j] > s[k]:
                count += j - i                 # mọi i' trong [i, j) đều thoả với j
                j -= 1
            else:
                i += 1
    return count

print("Số tam giác từ [2,2,3,4]:", count_triangles([2, 2, 3, 4]))

Số tam giác từ [2,2,3,4]: 3


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Kiểm tra ba cạnh
Viết `valid(a, b, c)` **không** dùng `sorted`, kiểm tra cả ba bất đẳng thức.

In [18]:
def valid(a: int, b: int, c: int) -> bool:
    # TODO
    pass

# assert valid(3, 4, 5) == True
# assert valid(1, 1, 2) == False
# assert valid(10, 1, 1) == False

### Bài 2 — Trả về bộ ba
Viết `best_triangle(nums)` trả về tuple `(a, b, c)` tăng dần của tam giác chu vi lớn nhất, hoặc `None`.

In [19]:
def best_triangle(nums: list[int]):
    # TODO
    pass

# assert best_triangle([3, 6, 2, 3]) == (2, 3, 3)
# assert best_triangle([1, 2, 1, 10]) is None

### Bài 3 — Tam giác cân
Trả về chu vi lớn nhất của tam giác **cân** (ít nhất hai cạnh bằng nhau), hoặc 0.

In [20]:
def largest_isosceles(nums: list[int]) -> int:
    # TODO
    pass

# assert largest_isosceles([5, 5, 9, 1]) == 19
# assert largest_isosceles([1, 2, 3]) == 0
# assert largest_isosceles([2, 2, 5]) == 0

### Bài 4 — Đếm tam giác (LeetCode 611)
Viết `count_valid(nums)` đếm số bộ ba **chỉ số** tạo tam giác, O(n²).

In [21]:
def count_valid(nums: list[int]) -> int:
    # TODO
    pass

# assert count_valid([2, 2, 3, 4]) == 3
# assert count_valid([4, 2, 3, 4]) == 4

### Bài 5 — Tam giác chu vi nhỏ nhất
Trả về chu vi **nhỏ nhất** của tam giác tạo được, hoặc 0.

In [22]:
def smallest_perimeter(nums: list[int]) -> int:
    # TODO
    pass

# assert smallest_perimeter([2, 3, 4, 100, 101, 102]) == 9
# assert smallest_perimeter([1, 1, 2, 10, 10, 2]) == 5

### Bài 6 (nâng cao) — Đa giác chu vi lớn nhất
Chọn **ít nhất 3** đoạn tạo đa giác (cạnh lớn nhất < tổng các cạnh còn lại), trả về chu vi lớn nhất hoặc `-1`.

In [23]:
def largest_polygon(nums: list[int]) -> int:
    # TODO
    pass

# assert largest_polygon([5, 5, 5]) == 15
# assert largest_polygon([1, 12, 1, 2, 5, 50, 3]) == 12
# assert largest_polygon([5, 5, 50]) == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | ba bất đẳng thức | dùng `and` | `a + b > c and a + c > b and b + c > a` |
| 2 | dùng lại lời giải chính | trả bộ thay vì tổng | `(s[i+2], s[i+1], s[i])` |
| 3 | hai cạnh bằng `x`, cạnh thứ ba `y` | điều kiện `2x > y`; `y` lấy từ các đoạn còn lại sau khi đã dùng hai đoạn `x` | thử mọi `x` có tần suất ≥ 2, chọn `y` lớn nhất hợp lệ |
| 4 | sắp xếp, cố định cạnh lớn | hai con trỏ `i`, `j` | `s[i]+s[j] > s[k]` → cộng `j − i` |
| 5 | cố định cạnh lớn `c = s[k]` | `b = s[k-1]` tốt nhất? cần `a` nhỏ nhất với `a > c − b` | duyệt cặp `(j, k)`, tìm `a` nhỏ nhất bằng chỉ số `0..j-1` |
| 6 | sắp tăng dần | tổng tiền tố | duyệt `i` từ cuối: `prefix[i] > s[i]` → trả `prefix[i] + s[i]` |

## 17. Đáp án tham khảo

In [24]:
from collections import Counter

def ans_valid(a, b, c):
    return a + b > c and a + c > b and b + c > a

def ans_best_triangle(nums):
    s = sorted(nums, reverse=True)
    for i in range(len(s) - 2):
        if s[i + 1] + s[i + 2] > s[i]:
            return (s[i + 2], s[i + 1], s[i])
    return None

def ans_largest_isosceles(nums):
    cnt = Counter(nums)
    best = 0
    for x, f in cnt.items():
        if f < 2:
            continue
        # cạnh thứ ba y: lấy từ phần còn lại (nếu y == x cần f >= 3), cần 0 < y < 2x
        rest = Counter(nums)
        rest[x] -= 2
        candidates = [y for y, g in rest.items() if g > 0 and y < 2 * x]
        if candidates:
            best = max(best, 2 * x + max(candidates))
    return best

def ans_count_valid(nums):
    s = sorted(nums)
    count = 0
    for k in range(len(s) - 1, 1, -1):
        i, j = 0, k - 1
        while i < j:
            if s[i] + s[j] > s[k]:
                count += j - i
                j -= 1
            else:
                i += 1
    return count

def ans_smallest_perimeter(nums):
    s = sorted(nums)
    best = 0
    for k in range(2, len(s)):
        for j in range(1, k):
            # a nhỏ nhất ở vị trí < j với s[a] > s[k] - s[j]
            for i in range(j):
                if s[i] + s[j] > s[k]:
                    p = s[i] + s[j] + s[k]
                    best = p if best == 0 else min(best, p)
                    break
    return best

def ans_largest_polygon(nums):
    s = sorted(nums)
    prefix = 0
    best = -1
    for i, x in enumerate(s):
        if i >= 2 and prefix > x:
            best = prefix + x
        prefix += x
    return best

assert ans_valid(3, 4, 5) == True
assert ans_valid(1, 1, 2) == False
assert ans_valid(10, 1, 1) == False
assert ans_best_triangle([3, 6, 2, 3]) == (2, 3, 3)
assert ans_best_triangle([1, 2, 1, 10]) is None
assert ans_largest_isosceles([5, 5, 9, 1]) == 19
assert ans_largest_isosceles([1, 2, 3]) == 0
assert ans_largest_isosceles([2, 2, 5]) == 0
assert ans_count_valid([2, 2, 3, 4]) == 3
assert ans_count_valid([4, 2, 3, 4]) == 4
assert ans_smallest_perimeter([2, 3, 4, 100, 101, 102]) == 9
assert ans_smallest_perimeter([1, 1, 2, 10, 10, 2]) == 5
assert ans_largest_polygon([5, 5, 5]) == 15
assert ans_largest_polygon([1, 12, 1, 2, 5, 50, 3]) == 12
assert ans_largest_polygon([5, 5, 50]) == -1

# Kiểm chứng thêm bài 5 bằng brute force
for _ in range(500):
    arr = [rng.randint(1, 20) for _ in range(rng.randint(3, 8))]
    per = [a + b + c for a, b, c in combinations(arr, 3) if is_triangle(a, b, c)]
    assert ans_smallest_perimeter(arr) == (min(per) if per else 0), arr
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def largestPerimeter(self, nums: list[int]) -> int:
        nums.sort(reverse=True)
        for i in range(len(nums) - 2):
            if nums[i + 1] + nums[i + 2] > nums[i]:
                return nums[i] + nums[i + 1] + nums[i + 2]
        return 0
```

Câu thần chú: **"Sắp giảm dần; hai cạnh sau cộng lại phải lớn hơn cạnh trước."**

**Checklist trước khi submit**
- [ ] Dùng `>` (không phải `>=`) để loại tam giác suy biến?
- [ ] So **hai cạnh nhỏ** với **cạnh lớn nhất**?
- [ ] Sắp giảm dần (hoặc tăng dần và duyệt ngược)?
- [ ] Vòng lặp dừng ở `len(nums) - 2`?
- [ ] Không có tam giác → `return 0`?
- [ ] Không viết `nums = nums.sort()`?